In [1]:
import requests
import pandas as pd

url_prefix = 'https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/03-evaluation/'
docs_url = url_prefix + 'search_evaluation/documents-with-ids.json'
documents = requests.get(docs_url).json()

ground_truth_url = url_prefix + 'search_evaluation/ground-truth-data.csv'
df_ground_truth = pd.read_csv(ground_truth_url)
ground_truth = df_ground_truth.to_dict(orient='records')

In [2]:
from tqdm.auto import tqdm

def hit_rate(relevance_total):
    cnt = 0

    for line in relevance_total:
        if True in line:
            cnt = cnt + 1

    return cnt / len(relevance_total)

def mrr(relevance_total):
    total_score = 0.0

    for line in relevance_total:
        for rank in range(len(line)):
            if line[rank] == True:
                total_score = total_score + 1 / (rank + 1)

    return total_score / len(relevance_total)

def evaluate(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        doc_id = q['document']
        results = search_function(q)
        relevance = [d['id'] == doc_id for d in results]
        relevance_total.append(relevance)

    return {
        'hit_rate': hit_rate(relevance_total),
        'mrr': mrr(relevance_total),
    }

/workspaces/llmzoomcamp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
documents[0]

{'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
 'section': 'General course-related questions',
 'question': 'Course - When will the course start?',
 'course': 'data-engineering-zoomcamp',
 'id': 'c02e79ef'}

In [4]:
ground_truth[0]

{'question': 'When does the course begin?',
 'course': 'data-engineering-zoomcamp',
 'document': 'c02e79ef'}

In [5]:
# Q1 
import minsearch

index = minsearch.Index(
    text_fields=["question", "text", "section"],
    keyword_fields=["course", "id"]
)

index.fit(documents)

def minsearch_search(query, course):
    boost = {'question': 1.5, 'section': 0.1}

    results = index.search(
        query=query,
        filter_dict={'course': course},
        boost_dict=boost,
        num_results=5
    )

    return results


In [6]:
relevance_total = []

for q in tqdm(ground_truth):
    doc_id = q['document']
    results = minsearch_search(query=q['question'], course=q['course'])
    relevance = [d['id'] == doc_id for d in results]
    relevance_total.append(relevance)

hit_rate(relevance_total)

# 0.848714069591528

100%|██████████| 4627/4627 [00:16<00:00, 272.18it/s]


0.848714069591528

In [7]:
# Q2
from minsearch import VectorSearch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import make_pipeline
texts = []

for doc in documents:
    t = doc['question']
    texts.append(t)

pipeline = make_pipeline(
    TfidfVectorizer(min_df=3),
    TruncatedSVD(n_components=128, random_state=1)
)
X = pipeline.fit_transform(texts)

In [8]:
vindex = VectorSearch(keyword_fields={'course'})
vindex.fit(X, documents)

In [9]:
def vector_search_function(query): 
    query_vector = pipeline.transform([query['question']])
    results = vindex.search(query_vector[0])
    return results

# 2. Evaluate using your evaluation function
metrics = evaluate(ground_truth, vector_search_function)
print(metrics)
#0.3003

100%|██████████| 4627/4627 [00:06<00:00, 676.75it/s]

{'hit_rate': 0.4696347525394424, 'mrr': 0.30038293179097125}


In [10]:
# Question 3
texts = []

for doc in documents:
    t = doc['question'] + ' ' + doc['text']
    texts.append(t)
pipeline = make_pipeline(
    TfidfVectorizer(min_df=3),
    TruncatedSVD(n_components=128, random_state=1)
)
X = pipeline.fit_transform(texts)
vindex = VectorSearch(keyword_fields={'course'})
vindex.fit(X, documents)

def vector_search_function(query): 
    query_vector = pipeline.transform([query['question']])
    results = vindex.search(query_vector[0])
    return results

# 2. Evaluate using your evaluation function
metrics = evaluate(ground_truth, vector_search_function)
print(metrics)


100%|██████████| 4627/4627 [00:09<00:00, 508.49it/s]

{'hit_rate': 0.8415820185865571, 'mrr': 0.6252495703273756}


In [30]:
# Question 4
from qdrant_client import QdrantClient, models 
from fastembed.embedding import TextEmbedding
client = QdrantClient("http://localhost:6333") # connect to local qdrant instance 
def embedding(model, query): 
    embedding_model = TextEmbedding(model_name=model)
    embeddings_generator = embedding_model.embed(query)
    embeddings_list = list(embeddings_generator)
    return embeddings_list

In [34]:
text = doc['question'] + ' ' + doc['text']
model_handle = "jinaai/jina-embeddings-v2-small-en"
limit = 5
collection_name = "llm-zoomcamp-homework-week3"

client.create_collection(
    collection_name = collection_name, 
    vectors_config=models.VectorParams(
        size = 512,
        distance = models.Distance.COSINE 
    )
)

True

In [35]:
points = [] 
id = 0 

for doc in documents: 
        point = models.PointStruct(
            id = id, 
            vector=models.Document(text=doc['question'] + ' ' + doc['text'],model = model_handle),  # embed locally 
            payload = {
                "text": doc['text'],
                "section": doc['section'],
                "course": doc['course']
            } # metadata
        )
        points.append(point)
        id += 1 

client.upsert(
    collection_name = collection_name,
    points=points
)
def search(query,limit=1):
    results = client.query_points(
        collection_name = collection_name,
        query=models.Document(
            text=query,
            model = model_handle
        ),
        limit = limit , 
        with_payload = True
    )
    return results 

: 

In [ ]:
evaluate(ground_truth, search)

In [12]:
# Q5 
import numpy as np 
def cosine(u, v):
    u_norm = np.sqrt(u.dot(u))
    v_norm = np.sqrt(v.dot(v))
    return u.dot(v) / (u_norm * v_norm)
results_url = url_prefix + 'rag_evaluation/data/results-gpt4o-mini.csv'
df_results = pd.read_csv(results_url)
pipeline = make_pipeline(
    TfidfVectorizer(min_df=3),
    TruncatedSVD(n_components=128, random_state=1)
)
pipeline.fit(df_results.answer_llm + ' ' + df_results.answer_orig + ' ' + df_results.question)



Pipeline(steps=[('tfidfvectorizer', TfidfVectorizer(min_df=3)),
                ('truncatedsvd',
                 TruncatedSVD(n_components=128, random_state=1))])

In [13]:
df_results.head() 

,answer_llm,answer_orig,document,question,course
0,You can sign up for the course by visiting the...,Machine Learning Zoomcamp FAQ\nThe purpose of ...,0227b872,Where can I sign up for the course?,machine-learning-zoomcamp
1,You can sign up using the link provided in the...,Machine Learning Zoomcamp FAQ\nThe purpose of ...,0227b872,Can you provide a link to sign up?,machine-learning-zoomcamp
2,"Yes, there is an FAQ for the Machine Learning ...",Machine Learning Zoomcamp FAQ\nThe purpose of ...,0227b872,Is there an FAQ for this Machine Learning course?,machine-learning-zoomcamp
3,The context does not provide any specific info...,Machine Learning Zoomcamp FAQ\nThe purpose of ...,0227b872,Does this course have a GitHub repository for ...,machine-learning-zoomcamp
4,To structure your questions and answers for th...,Machine Learning Zoomcamp FAQ\nThe purpose of ...,0227b872,How can I structure my questions and answers f...,machine-learning-zoomcamp


In [14]:
# Transform answers
X_llm = pipeline.transform(df_results['answer_llm'])
X_orig = pipeline.transform(df_results['answer_orig'])
cosine_scores = [cosine(u, v) for u, v in zip(X_llm, X_orig)]

# Compute average
avg_cosine_similarity = np.mean(cosine_scores)
avg_cosine_similarity
# 0.841584123349

np.float64(0.8415841233490402)

In [23]:
# Q6
from rouge import Rouge
rouge_scorer = Rouge()

result = [] 
for _, row in df_results.iterrows():
    scores = rouge_scorer.get_scores(row['answer_llm'], row['answer_orig'])[0]['rouge-1']['f']
    result.append(scores)

In [ ]:
np.mean(result)
# 0.351694

np.float64(0.3516946452113943)